# 🧪 **Taller de Clustering: K-Means, K-Medians, K-Medoids, DBSCAN y criterio del codo**

### 🔜 **Introducción**

En clustering no existe un único método que funcione mejor en todos los contextos. Algunos algoritmos buscan centros promedio, otros minimizan distancias absolutas, otros seleccionan representantes reales del conjunto y otros detectan regiones densas. Por eso, en este taller se trabajará con varios datasets y se pedirá experimentar, visualizar, responder preguntas y sacar conclusiones.

##### 🎯 **Objetivos del taller**

- Aplicar el criterio del codo para proponer valores razonables de `k`.
- Implementar y comparar **K-Means**, **K-Medians** y **K-Medoids**.
- Analizar la sensibilidad de los métodos ante outliers, formas no convexas y cambios de parámetros.
- Explorar **DBSCAN** variando `eps` y `min_samples`.
- Argumentar decisiones metodológicas a partir de gráficas, métricas y observaciones cualitativas.

##### 🧭 **¿Qué veremos?**

El taller está dividido en varias estaciones. En cada una se propone un problema distinto: elegir `k` con el criterio del codo, comparar clustering por medias, medianas y medoides, estudiar el impacto de outliers y trabajar con clustering por densidad en datos no lineales. Cada sección incluye preguntas conceptuales y ejercicios prácticos.

##### 📌 **Instrucciones generales**

1. Ejecuta las celdas base y completa las celdas marcadas como ejercicio.
2. No te limites a correr el código una sola vez: modifica parámetros, repite experimentos y documenta observaciones.
3. Cuando se te pida justificar una respuesta, apóyate en visualizaciones, métricas y comportamiento del algoritmo.
4. En varios ejercicios no hay una única respuesta correcta; importa la calidad del análisis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

### 🧰 **Funciones auxiliares**

Las siguientes funciones sirven para generar gráficas y calcular algunas cantidades útiles. No es necesario modificarlas al comienzo del taller, pero puedes reutilizarlas y adaptarlas.

In [ ]:
def plot_clusters(X, labels=None, centers=None, title='Gráfica'):
    plt.figure(figsize=(7, 5))
    if labels is None:
        plt.scatter(X[:, 0], X[:, 1], s=35)
    else:
        plt.scatter(X[:, 0], X[:, 1], c=labels, s=35)
    if centers is not None:
        plt.scatter(centers[:, 0], centers[:, 1], marker='X', s=220)
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()

def inertia_manual(X, labels, centers):
    total = 0.0
    for j in range(len(centers)):
        cluster_points = X[labels == j]
        if len(cluster_points) > 0:
            total += np.sum((cluster_points - centers[j]) ** 2)
    return total

def l1_cost(X, labels, centers):
    total = 0.0
    for j in range(len(centers)):
        cluster_points = X[labels == j]
        if len(cluster_points) > 0:
            total += np.sum(np.abs(cluster_points - centers[j]))
    return total

def pairwise_distances_numpy(X, Y=None, metric='euclidean'):
    if Y is None:
        Y = X
    if metric == 'euclidean':
        return np.sqrt(((X[:, None, :] - Y[None, :, :]) ** 2).sum(axis=2))
    elif metric == 'manhattan':
        return np.abs(X[:, None, :] - Y[None, :, :]).sum(axis=2)
    else:
        raise ValueError('Métrica no soportada')

### 📉 **Parte 1: K-Means y criterio del codo**

Comenzaremos con un dataset relativamente favorable para K-Means. La idea es practicar selección de `k`, interpretación de inercia y análisis de calidad de partición.

In [ ]:
np.random.seed(7)
X_blobs, y_blobs = make_blobs(
    n_samples=450,
    centers=[(-5, -2), (0, 0), (4, 5), (7, -3)],
    cluster_std=[1.0, 1.3, 0.8, 1.1],
    random_state=7
)

plot_clusters(X_blobs, title='Dataset 1: blobs para K-Means')

##### 🧪 **Ejercicio 1: explorar varios valores de** `k`

Entrena modelos de K-Means para varios valores de `k` entre 2 y 8. Grafica la inercia y, si lo consideras útil. A partir de la curva, propón un valor razonable de `k`.

In [ ]:
# Escribe tu solución aquí
# Sugerencia:
# 1. Recorre varios valores de k
# 2. Guarda la inercia
# 3. Grafica los resultados

##### ❓ **Preguntas de análisis 1**

1. ¿En qué valor de `k` observas el codo más claro?
2. ¿Coincide ese valor con lo que sugiere la visualización del dataset?
3. ¿Qué pasa con la inercia cuando `k` aumenta? ¿Por qué esto impide elegir `k` sólo por minimizar inercia?
4. ¿En qué situaciones reales el criterio del codo puede resultar ambiguo?

##### 🧪 **Ejercicio 2: inspección geométrica de K-Means**

Ajusta K-Means con el valor de `k` que consideres más razonable. Visualiza los clusters y los centroides. Luego prueba con un `k` menor y con un `k` mayor para comparar.

In [ ]:
# Escribe tu solución aquí

##### ❓ **Preguntas de análisis 2**

1. ¿Qué efecto produce subestimar `k`?
2. ¿Qué efecto produce sobreestimar `k`?
3. ¿Siempre el mejor `k` desde la métrica coincide con el más interpretable visualmente?

### 🧱 **Parte 2: K-Medians frente a K-Means con outliers**

Ahora construiremos un dataset donde la presencia de outliers pueda afectar fuertemente el cálculo de centroides. La pregunta central será: ¿qué cambia si, en lugar de medias, usamos medianas por coordenada?

In [ ]:
np.random.seed(15)
X_core, _ = make_blobs(
    n_samples=240,
    centers=[(-3, -3), (3, 3), (6, -2)],
    cluster_std=[0.9, 0.9, 0.9],
    random_state=15
)

outliers = np.array([
    [12, 12], [13, 11], [14, 13],
    [-10, 10], [-11, 9], [11, -9], [12, -10]
])

X_out = np.vstack([X_core, outliers])
plot_clusters(X_out, title='Dataset 2: blobs con outliers')

##### 🛠️ **Implementación base de K-Medians**

A diferencia de K-Means, en **K-Medians** la actualización de los centros se hace usando la mediana componente a componente, y la asignación suele apoyarse en distancia Manhattan. La siguiente implementación puede servir como punto de partida para tus análisis.

In [ ]:
def kmedians(X, k, max_iter=100, random_state=0):
    rng = np.random.default_rng(random_state)
    centers = X[rng.choice(len(X), size=k, replace=False)].copy()

    for _ in range(max_iter):
        distances = pairwise_distances_numpy(X, centers, metric='manhattan')
        labels = np.argmin(distances, axis=1)

        new_centers = centers.copy()
        for j in range(k):
            cluster_points = X[labels == j]
            if len(cluster_points) > 0:
                new_centers[j] = np.median(cluster_points, axis=0)

        if np.allclose(new_centers, centers):
            break
        centers = new_centers

    distances = pairwise_distances_numpy(X, centers, metric='manhattan')
    labels = np.argmin(distances, axis=1)
    return labels, centers

##### 🧪 **Ejercicio 3: comparar K-Means y K-Medians**

Ajusta ambos métodos con el mismo valor de `k` sobre `X_out`. Visualiza los resultados y compara la ubicación de los centros.

In [ ]:
# Escribe tu solución aquí
# Sugerencia: usa k=3 como punto de partida

##### ❓ **Preguntas de análisis 3**

1. ¿Cuál de los dos métodos parece más estable frente a los outliers?
2. ¿Qué diferencia conceptual hay entre minimizar suma de distancias cuadráticas y suma de distancias absolutas?
3. ¿En qué tipo de problema esperarías que K-Medians fuera preferible a K-Means?
4. ¿La frontera entre clusters cambia al pasar de distancia euclidiana a Manhattan?

##### 🧪 **Ejercicio 4: experimento controlado con y sin outliers**

Repite el análisis anterior usando `X_core` y luego `X_out`. Compara cómo cambian los centros de K-Means y K-Medians cuando se agregan outliers.

In [ ]:
# Escribe tu solución aquí

### 🪨 **Parte 3: K-Medoids y representantes reales del dataset**

En **K-Medoids**, cada centro debe ser un punto real del dataset. Este detalle puede hacer al método más interpretable y más robusto en algunos contextos. En esta sección trabajarás con una implementación sencilla basada en actualización por costo mínimo.

In [ ]:
np.random.seed(21)
X_medoid, _ = make_blobs(
    n_samples=180,
    centers=[(-4, 0), (0, 4), (5, 1)],
    cluster_std=[1.1, 0.9, 1.2],
    random_state=21
)

X_medoid = np.vstack([
    X_medoid,
    np.array([[8, 8], [9, 7], [-8, -4]])
])

plot_clusters(X_medoid, title='Dataset 3: datos para K-Medoids')

##### 🛠️ **Implementación base de K-Medoids**

Esta implementación usa distancia euclidiana, pero podrías modificarla para distancia Manhattan y observar diferencias.

In [ ]:
def kmedoids(X, k, max_iter=100, random_state=0, metric='euclidean'):
    rng = np.random.default_rng(random_state)
    medoid_indices = rng.choice(len(X), size=k, replace=False)
    medoids = X[medoid_indices].copy()

    for _ in range(max_iter):
        distances = pairwise_distances_numpy(X, medoids, metric=metric)
        labels = np.argmin(distances, axis=1)

        new_medoid_indices = medoid_indices.copy()

        for j in range(k):
            cluster_indices = np.where(labels == j)[0]
            if len(cluster_indices) == 0:
                continue
            cluster_points = X[cluster_indices]
            D = pairwise_distances_numpy(cluster_points, cluster_points, metric=metric)
            best_local = np.argmin(D.sum(axis=1))
            new_medoid_indices[j] = cluster_indices[best_local]

        if np.array_equal(new_medoid_indices, medoid_indices):
            break

        medoid_indices = new_medoid_indices
        medoids = X[medoid_indices].copy()

    distances = pairwise_distances_numpy(X, medoids, metric=metric)
    labels = np.argmin(distances, axis=1)
    return labels, medoids, medoid_indices

##### 🧪 **Ejercicio 5: comparar K-Means y K-Medoids**

Ajusta ambos métodos con el mismo valor de `k` y compara la posición de los centros. Verifica explícitamente que los medoides son observaciones reales del dataset.

In [ ]:
# Escribe tu solución aquí

##### ❓ **Preguntas de análisis 4**

1. ¿Qué ventaja interpretativa tiene que un centro sea una observación real?
2. ¿Por qué K-Medoids suele ser más costoso computacionalmente que K-Means?
3. ¿Qué efecto tendrían distancias no euclidianas en K-Medoids?
4. ¿En qué contexto aplicado preferirías K-Medoids sobre K-Means?

##### 🧪 **Ejercicio 6: cambiar la métrica en K-Medoids**

Ejecuta K-Medoids con distancia euclidiana y luego con distancia Manhattan. Compara la asignación de clusters y la identidad de los medoides.

In [ ]:
# Escribe tu solución aquí

### 🌙 **Parte 4: DBSCAN en datos no convexos**

Ahora cambiaremos de paradigma. En datasets con forma curva o no convexa, K-Means, K-Medians y K-Medoids pueden producir particiones poco naturales. DBSCAN intenta resolver esto buscando conectividad por densidad.

In [ ]:
np.random.seed(42)
X_moons, y_moons = make_moons(n_samples=350, noise=0.08, random_state=42)
plot_clusters(X_moons, title='Dataset 4: dos lunas')

##### 🧪 **Ejercicio 7: contraste entre K-Means y DBSCAN**

Ajusta K-Means con `k=2` y luego DBSCAN sobre el mismo dataset. Prueba varios valores de `eps` y `min_samples`.

In [ ]:
# Escribe tu solución aquí

##### ❓ **Preguntas de análisis 5**

1. ¿Por qué K-Means no logra representar bien la estructura de las lunas?
2. ¿Qué ocurre cuando `eps` es demasiado pequeño?
3. ¿Qué ocurre cuando `eps` es demasiado grande?
4. ¿Cómo influye `min_samples` en la cantidad de puntos marcados como ruido?
5. ¿Qué ventaja ofrece DBSCAN al no exigir especificar `k`?

##### 🧪 **Ejercicio 8: mapa de sensibilidad de parámetros en DBSCAN**

Construye una tabla donde registres, para varias combinaciones de `eps` y `min_samples`, el número de clusters encontrados y la cantidad de ruido. Luego interpreta los resultados.

In [ ]:
# Escribe tu solución aquí

### 🌀 **Parte 5: un dataset adicional para poner a prueba decisiones metodológicas**

En esta sección trabajarás con un dataset diferente para que decidas cuál algoritmo tiene más sentido. La idea no es sólo ejecutar modelos, sino defender una elección.

In [ ]:
X_circles, y_circles = make_circles(n_samples=400, factor=0.45, noise=0.05, random_state=11)
plot_clusters(X_circles, title='Dataset 5: círculos concéntricos')

##### 🧪 **Ejercicio 9: selección del método**

Aplica al menos dos de los siguientes métodos sobre `X_circles`: K-Means, K-Medians, K-Medoids, DBSCAN. Visualiza y argumenta cuál se adapta mejor a la estructura del dataset.

In [ ]:
# Escribe tu solución aquí

##### ❓ **Preguntas de análisis 6**

1. ¿Hay algún método basado en centros que funcione bien en círculos concéntricos? Justifica.
2. ¿Qué tipo de geometría del dataset favorece a DBSCAN?
3. ¿Qué papel jugaría el escalamiento de variables en este ejercicio?

### 🧠 **Parte 6: preguntas integradoras**

Responde de forma argumentada. Puedes apoyar tus respuestas con ejemplos de este mismo taller.

##### ❓ **Preguntas finales**

1. Compara los objetivos de optimización de K-Means, K-Medians y K-Medoids.
2. Explica por qué el criterio del codo no aplica directamente a DBSCAN.
3. ¿Puede un valor grande de `k` producir clusters aparentemente buenos pero poco útiles?
4. ¿Qué algoritmo elegirías si el dataset tiene outliers severos pero clusters aproximadamente compactos?
5. ¿Qué algoritmo elegirías si los clusters tienen forma arbitraria y además hay ruido?
6. ¿Por qué la interpretación visual sigue siendo importante incluso cuando se usan métricas?

### 🏁 **Reto final**

Elige uno de los datasets del taller y realiza un miniinforme dentro del notebook. Tu informe debe incluir: problema planteado, algoritmo(s) probados, parámetros explorados, evidencia gráfica, métricas usadas y una conclusión razonada. La calidad del argumento será tan importante como el resultado numérico.

In [ ]:
# Escribe aquí tu miniinforme final en formato libre.
# Puedes combinar texto, tablas, gráficas y conclusiones.